# 2. Очистка и проверка результата

Очистка реализована в [clean_data.py](clean_data.py). Тетрадь показывает последовательность, правила, журнал исправлений и контрольные итоги. Скрипт запускается самостоятельно на исходном CSV.

**Порядок:** сначала проверяем повторы исходных записей и используем модификацию для исправления гибридов; затем удаляем ненужные поля. Это осознанное уточнение порядка задания, предотвращающее потерю информации при очистке.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Тетради можно открывать из папки проекта или из coursework.
ROOT = Path.cwd() if Path('clean_data.py').exists() else Path.cwd() / 'coursework'
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
from clean_data import load_raw, normalize, aggregate_clean, read_clean, run, validate, CATEGORIES, DROP, COMPANIES, MONTHS
pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

raw = load_raw('data/autokz2019_raw.csv')
detailed, quarantine, corrections, unresolved, meta = normalize(raw)
display(pd.Series(meta))

input_rows                      39,966.00
empty_rows                           0.00
exact_duplicates_removed        18,698.00
rows_after_exact_dedup          21,268.00
quarantine_rows                      6.00
valid_source_rows               21,262.00
renault_shift_rows                  27.00
niva_corrected_rows                 25.00
source_quantity_raw             53,874.00
quantity_after_cleaning         35,165.00
sales_after_cleaning_usd   815,179,587.19
dtype: float64

## Соответствие названий заданию
`Дилер` → «Наименование дилерского центра»; `Тип платежа` → «Форма расчета»; `Локализация` → «Локализация производства». Буквального поля «Сегмент ЕПР» в CSV нет: удалено дополнительное поле «Сегментация Eng» как ближайший аналог по назначению. «Сегментация 2013» и «Класс 2013» сохранены.

«Стоимость» → `unit_price_usd`, «Количество» → `quantity`, «Итоговая стоимость» → `total_sales_usd`. Все итоговые имена английские.

In [2]:
from clean_data import RENAME, COUNTRIES
display(pd.DataFrame({'Исходное поле': list(RENAME), 'Английское имя': list(RENAME.values())}))
display(pd.Series(COUNTRIES, name='ALPHA-3').to_frame())

,Исходное поле,Английское имя
0,Год,year
1,Месяц,month
2,Компания,company
3,Бренд,brand
4,Модель,model
5,Модификация,modification
6,Год выпуска,manufacture_year
7,Страна-производитель,country
8,Вид топлива,fuel_type
9,"Объём двиг, л,",engine_l


,ALPHA-3
Республика Казахстан,KAZ
Российская Федерация,RUS
Япония,JPN
Турция,TUR
Таиланд,THA
США,USA
Германия,DEU
UK,GBR
Китай,CHN
Узбекистан,UZB


## Правила нормализации

- Страны: явный словарь ALPHA-3; «Корея» трактуется как KOR в контексте автомобильных брендов источника. Неизвестная новая страна останавливает скрипт.
- Топливо: F, D, E, HYB. Обозначение `E-Hybrid`/`HYBR` в модификации имеет приоритет над ошибочным бензином.
- Привод: 4WD, FWD, RWD, 2WD. У 2WD неизвестна ведущая ось; приравнивать его к FWD нельзя. `4х2.2` трактуется как двухколёсный привод без определения оси; «Астана» и `0` становятся пропусками.
- Коробка: «Автомат» включает вариатор, робот и электрический редуктор; «Механика» — механическую коробку. TDI не обозначает коробку: оставлен пропуск. Для Cayenne число 8 классифицировано по согласующимся распознанным строкам модели; это документированное восстановление.
- Renault: из смещённых столбцов восстановлены объём и коробка; утраченное топливо не угадывается.
- Двигатель: десятичная точка, удаление суффиксов L/MPI/GDI и др.; Niva исправлена на 1.7 по устойчивому значению остальных строк и последовательности ошибки. У электромобилей объём ДВС 0; мощность и ёмкость батареи не считаются литрами.
- Год выпуска: удалены неразрывные пробелы, тип Int64 допускает пропуски. Регионы и области приведены к `title()`; исторические названия не заменены современными.
- Дата: календарный конец месяца, включая 28 февраля; исходные год и месяц удаляются.

In [3]:
display(corrections.groupby(['field', 'reason']).size().rename('rows').reset_index())
display(corrections.head(12))
display(unresolved)
print('Исправленные значения объёма Niva:')
display(detailed.loc[detailed.brand.eq('Chevrolet') & detailed.model.eq('Niva'), 'engine_l'].value_counts())

,field,reason,rows
0,engine_l,Chevrolet Niva: последовательная ошибка 2.7–26...,25
1,engine_l,Объём восстановлен из поля топлива Renault,27
2,engine_l,Электромобиль: рабочий объём ДВС отсутствует; ...,5
3,fuel_type,Исходный тип топлива при сдвиге утрачен; не уг...,27
4,fuel_type,Модель Jaguar I-Pace: остальные строки явно об...,1
5,fuel_type,Явное обозначение гибрида в модификации,6
6,transmission,Восстановление сдвига полей Renault,27
7,transmission,Число 8 — число ступеней; все распознанные Cay...,5
8,transmission,Электропривод отнесён к автомату в требуемой д...,5


,source_row,field,before,after,reason
0,23725,transmission,4WD,MT,Восстановление сдвига полей Renault
1,23726,transmission,4WD,AT,Восстановление сдвига полей Renault
2,23728,transmission,4WD,AT,Восстановление сдвига полей Renault
3,23729,transmission,4WD,MT,Восстановление сдвига полей Renault
4,23730,transmission,4WD,MT,Восстановление сдвига полей Renault
5,23731,transmission,4WD,AT,Восстановление сдвига полей Renault
6,23733,transmission,4WD,MT,Восстановление сдвига полей Renault
7,23735,transmission,4WD,MT,Восстановление сдвига полей Renault
8,23736,transmission,Передний,AT,Восстановление сдвига полей Renault
9,23748,transmission,4WD,AT,Восстановление сдвига полей Renault


,field,unresolved_token,rows
0,fuel_type,0,3
1,transmission,TDI,3
2,drive_type,0,68
3,drive_type,Астана,2


Исправленные значения объёма Niva:


engine_l
1.70    279
Name: count, dtype: Int64

## Пропуски и правдоподобные аномалии
Шесть записей без количества исключены из готовой аналитической таблицы и сохранены в `quarantine.csv`. Нулевая сумма не доказывает нулевое количество. Остальные неизвестные характеристики сохранены как пропуски: среднее или мода могли бы приписать автомобилю неверную комплектацию. В разрезах EDA они показаны как «Неизвестно».

Отрицательное количество сохраняется как возврат. Партии 100 Skoda и 115 Lada сохраняются. Большие двигатели коммерческой техники не удаляются по порогу, подходящему легковым машинам. Значение 2.5 л у автобусов Daewoo вызывает вопрос, но внутреннего подтверждения правильного объёма нет; оно оставлено без выдуманной замены и учитывается как ограничение анализа объёма двигателя.

In [4]:
display(quarantine[['source_row', 'brand', 'model', 'quantity', 'total_sales_usd', 'reason']])
display(detailed.loc[detailed.quantity.lt(0) | detailed.quantity.ge(100),
                     ['source_row', 'brand', 'model', 'quantity', 'unit_price_usd', 'total_sales_usd']])

,source_row,brand,model,quantity,total_sales_usd,reason
16741,16743,Kia,Sportage,<NA>,0.00,Неизвестное количество или некорректное числов...
17368,17370,Lada,Granta,<NA>,0.00,Неизвестное количество или некорректное числов...
18676,18678,Lada,XRAY,<NA>,0.00,Неизвестное количество или некорректное числов...
37416,37418,UAZ,3909,<NA>,0.00,Неизвестное количество или некорректное числов...
37727,37729,UAZ,Patriot,<NA>,0.00,Неизвестное количество или некорректное числов...
38775,38777,Skoda,Kodiaq,<NA>,0.00,Неизвестное количество или некорректное числов...


,source_row,brand,model,quantity,unit_price_usd,total_sales_usd
16885,16887,Lada,4x4,115,"9,000.00","1,035,000.00"
25555,25557,Skoda,Octavia,100,"18,700.00","1,870,000.00"
25728,25730,Skoda,Superb,-1,"35,588.25","-35,588.25"


## Удаление полей и объединение совпадающих аналитических групп
После нормализации и удаления семи полей строки с одинаковыми оставшимися характеристиками, ценой, месяцем и признаком возврата суммируются. Это **агрегация, а не повторное удаление продаж**. `source_records` сохраняет число представленных исходных записей. Возвраты не смешиваются с положительными продажами.

Итоговое зерно: месяц × компания × характеристики автомобиля × регион/область × цена × признак возврата. Поле `quantity` остаётся весом строки.

In [5]:
clean = aggregate_clean(detailed)
display(pd.DataFrame({'До агрегации': [len(detailed), detailed.quantity.sum(), detailed.total_sales_usd.sum()],
                      'После агрегации': [len(clean), clean.quantity.sum(), clean.total_sales_usd.sum()]},
                     index=['Строк', 'Автомобилей нетто', 'Продажи, USD']))
display(clean.dtypes.astype(str).to_frame('dtype'))
display(clean.isna().sum().to_frame('missing_rows'))

,До агрегации,После агрегации
Строк,"21,262.00","16,141.00"
Автомобилей нетто,"35,165.00","35,165.00"
"Продажи, USD","815,179,587.19","815,179,587.19"


,dtype
company,string
brand,string
model,string
manufacture_year,Int64
country,str
fuel_type,category
engine_l,Float64
transmission,category
drive_type,category
region,string


,missing_rows
company,0
brand,0
model,0
manufacture_year,184
country,0
fuel_type,1074
engine_l,1384
transmission,1005
drive_type,1356
region,0


## Экспорт и повторное чтение
CSV не хранит category/datetime: функция `read_clean()` восстанавливает типы. Контролируем также суммы после повторного чтения.

In [6]:
clean, detailed, summary = run('data/autokz2019_raw.csv', 'data')
restored = read_clean('data/autokz2019_clean.csv')
pd.testing.assert_frame_equal(clean.reset_index(drop=True), restored.reset_index(drop=True),
                              check_dtype=False, check_categorical=False, rtol=1e-10, atol=1e-6)
validate(restored)
assert not set(DROP + ['year', 'month']) & set(restored.columns)
assert set(restored.sale_date.dt.month) == set(range(1, 10))
assert restored.loc[restored.sale_date.dt.month.eq(2), 'sale_date'].dt.day.eq(28).all()
assert all(isinstance(restored[c].dtype, pd.CategoricalDtype) for c in CATEGORIES)
print('Экспорт, повторное чтение, календарь, категории и контрольные суммы проверены.')
display(pd.Series(summary))

Экспорт, повторное чтение, календарь, категории и контрольные суммы проверены.


input_rows                                                              39966
empty_rows                                                                  0
exact_duplicates_removed                                                18698
rows_after_exact_dedup                                                  21268
quarantine_rows                                                             6
valid_source_rows                                                       21262
renault_shift_rows                                                         27
niva_corrected_rows                                                        25
source_quantity_raw                                                 53,874.00
quantity_after_cleaning                                                 35165
sales_after_cleaning_usd                                       815,179,587.19
output_rows                                                             16141
output_columns                                                  

## Вывод этапа
Получено 16 141 аналитических строк из 21 262 пригодных исходных записей после удаления точных повторов и карантина. Сохранено 35 165 автомобилей нетто, включая экспорт. Все требуемые преобразования выполнены; неизвестные характеристики не подменены догадками. Протокол исправлений, карантин, схема и промежуточные записи сохранены рядом с итоговым CSV.

**Сверка:** семь лишних полей удалены, английские названия присвоены, повторы обработаны, страны/топливо/привод/компании/двигатель/география/коробки нормализованы, числа и даты приведены к нужным типам. Ограничения: нет идентификатора сделки, часть технических характеристик неизвестна, «Сегмент ЕПР» сопоставлен по смыслу.